In [1]:
import tifffile
import dask.array as da
import napari

# UPDATE THESE PATHS
image_path = "/Volumes/h30492/farkkilab2/9_EyeMT/Data/cycif/batch1_adjacent_slides/tif/run/S053_iOme.ome.tif"
mask_path = "/Volumes/h30492/farkkilab2/9_EyeMT/Data/cycif/batch1_adjacent_slides/seg/S053_iOme.ome.tif"

# SELECT YOUR CHANNELS (you have 0-12 available)
channels_to_load = [0, 5, 9]

print("Loading image pyramid...")
image_pyramid = []
for level in range(7):  # 7 levels
    # Load all channels at this level, then select yours
    level_data = da.from_zarr(tifffile.imread(image_path, aszarr=True, level=level))
    level_data = level_data[channels_to_load]  # Select channels
    image_pyramid.append(level_data)
    print(f"  Level {level}: {level_data.shape}")

print("\nLoading mask pyramid...")
mask_pyramid = []
for level in range(5):  # 5 levels
    mask_data = da.from_zarr(tifffile.imread(mask_path, aszarr=True, level=level))
    mask_pyramid.append(mask_data)
    print(f"  Level {level}: {mask_data.shape}")

print("\nStarting napari...")
viewer = napari.Viewer()

for i, ch_idx in enumerate(channels_to_load):
    ch_pyramid = [level[i] for level in image_pyramid]
    viewer.add_image(ch_pyramid, name=f"Channel {ch_idx}", multiscale=True)

viewer.add_labels([m.astype('uint32') for m in mask_pyramid], name="Nuclei", opacity=0.3, multiscale=True)

print("✓ Ready! Zoom in/out with scroll wheel, pan with middle-click + drag")
napari.run()

Loading image pyramid...
  Level 0: (3, 66298, 53139)
  Level 1: (3, 33149, 26570)
  Level 2: (3, 16575, 13285)
  Level 3: (3, 8288, 6643)
  Level 4: (3, 4144, 3322)
  Level 5: (3, 2072, 1661)
  Level 6: (3, 1036, 831)

Loading mask pyramid...
  Level 0: (66298, 53139)
  Level 1: (33149, 26569)
  Level 2: (16574, 13284)
  Level 3: (8287, 6642)
  Level 4: (4143, 3321)

Starting napari...
✓ Ready! Zoom in/out with scroll wheel, pan with middle-click + drag
